# e-stat API取得

`.env` に `E_STAT_APPID`（または `APPID`）を置いておき、以下のセルで読み込み・API呼び出しを行います。

In [28]:
# 必要ライブラリの読み込み
from dotenv import load_dotenv
import os
import requests
import json

# .env を読み込む（事前に .env に E_STAT_APPID=xxxxx を設定しておく）
load_dotenv()
app_id = os.getenv('E_STAT_APPID') or os.getenv('APPID') or ''
if not app_id:
    raise EnvironmentError('`.env` に E_STAT_APPID または APPID を設定してください')

# ベース URL とパラメータを分けて渡す（requests は str を受け取る）
base_url = 'http://api.e-stat.go.jp/rest/3.0/app/json/getStatsData'
params = {
    'appId': app_id,
    'lang': 'J',
    'statsDataId': '0003045904',
    'metaGetFlg': 'Y',
    'cntGetFlg': 'N',
    'explanationGetFlg': 'Y',
    'annotationGetFlg': 'Y',
    'sectionHeaderFlg': '1',
    'replaceSpChars': '0',
    'cdCatSelect1': 'C01',
    'cdArea': '01000',
}

# リクエスト実行
try:
    resp = requests.get(base_url, params=params, timeout=20)
    resp.raise_for_status()
    data = resp.json()
except requests.RequestException as e:
    raise SystemExit(f'API request failed: {e}')

# サマリ表示（必要に応じて詳細は data を確認）
print('取得したトップレベルキー:', list(data.keys()))
# 主要部分を見やすく表示
print(json.dumps(data, ensure_ascii=False)[:2000])

取得したトップレベルキー: ['GET_STATS_DATA']
{"GET_STATS_DATA": {"RESULT": {"STATUS": 0, "ERROR_MSG": "正常に終了しました。", "DATE": "2026-06-24T23:19:46.888+09:00"}, "PARAMETER": {"LANG": "J", "STATS_DATA_ID": "0003045904", "NARROWING_COND": {"CODE_AREA_SELECT": "01000"}, "DATA_FORMAT": "J", "START_POSITION": 1, "METAGET_FLG": "Y", "EXPLANATION_GET_FLG": "Y", "ANNOTATION_GET_FLG": "Y", "REPLACE_SP_CHARS": 0, "CNT_GET_FLG": "N", "SECTION_HEADER_FLG": 1}, "STATISTICAL_DATA": {"RESULT_INF": {"TOTAL_NUMBER": 576, "FROM_NUMBER": 1, "TO_NUMBER": 576}, "TABLE_INF": {"@id": "0003045904", "STAT_NAME": {"@code": "00351000", "$": "民間給与実態統計調査"}, "GOV_ORG": {"@code": "00351", "$": "国税庁"}, "STATISTICS_NAME": "民間給与実態統計 結果表", "TITLE": {"@no": "00101", "$": "全国計表　第1表　給与所得者数・給与額・税額 事業所規模別 （2007年～2014年）"}, "CYCLE": "年次", "SURVEY_DATE": 0, "OPEN_DATE": "2015-11-19", "SMALL_AREA": 0, "COLLECT_AREA": "該当なし", "MAIN_CATEGORY": {"@code": "03", "$": "労働・賃金"}, "SUB_CATEGORY": {"@code": "02", "$": "賃金・労働条件"}, "OVERALL_TOTAL_NUMBER":

In [29]:
import pandas as pd
df = pd.DataFrame(data['GET_STATS_DATA']['STATISTICAL_DATA']['DATA_INF']['VALUE'])
display(df)

,@tab,@cat01,@time,@unit,$
0,0010,1,2014000000,人,9825309
1,0010,1,2013000000,人,10025169
2,0010,1,2012000000,人,9713060
3,0010,1,2011000000,人,10010414
4,0010,1,2010000000,人,10303333
...,...,...,...,...,...
571,0090,9,2011000000,千円,148
572,0090,9,2010000000,千円,137
573,0090,9,2009000000,千円,138
574,0090,9,2008000000,千円,157


In [11]:
import pandas as pd
df = pd.DataFrame(data['GET_STATS_DATA']['STATISTICAL_DATA']['DATA_INF']['VALUE'])
display(df)

,@tab,@cat01,@area,@time,@unit,$
0,20200,102,00000,2020100000,人,672323
1,20200,102,01000,2020100000,人,32503
2,20200,102,02000,2020100000,人,8978
3,20200,102,03000,2020100000,人,8754
4,20200,102,04000,2020100000,人,12307
...,...,...,...,...,...,...
3063,20220,273,33100,2020100000,人口10万対,1102.3
3064,20220,273,34100,2020100000,人口10万対,998.1
3065,20220,273,40100,2020100000,人口10万対,1207.2
3066,20220,273,40130,2020100000,人口10万対,1167.3


In [37]:
"""
Utility for calling e-stat '統計表情報取得' (getStatsList) API (仕様 v3.0).

Parameters (日本語説明):
  - app_id: アプリケーションID（必須）。`.env` に `E_STAT_APPID` または `APPID` を設定してください。
  - lang: 取得するデータの言語。'J'（日本語, 省略値）または 'E'（英語）。
  - surveyYears: 調査年月。'yyyy'、'yyyymm'、または 'yyyymm-yyyymm' の形式。
  - openYears: 公開年月。surveyYears と同様の形式。
  - statsField: 統計分野コード（数値2桁/4桁）。
  - statsCode: 政府統計コード（作成機関5桁/政府統計コード8桁）。
  - statsNameList: 統計調査名一覧指定。'Y' を指定すると統計調査名一覧を返す。
  - searchWord: 検索キーワード。AND/OR/NOT を使った複合検索も可（例: '東京 AND 人口'）。
  - searchKind: 検索データ種別。'1'=統計情報（省略値）, '2'=小地域・地域メッシュ。
  - collectArea: 集計地域区分。'1'=全国, '2'=都道府県, '3'=市区町村。
  - explanationGetFlg: 解説情報有無。'Y'=取得（省略値）, 'N'=取得しない。
  - startPosition: データ取得開始位置（1始まり）。継続取得時に使用。
  - limit: 取得件数。省略時はサービス側のデフォルト（例: 統計表検索は100000等）。
  - updatedDate: 更新日付。'yyyy'、'yyyymm'、'yyyymmdd'、または範囲指定。
  - dataFormat: 出力形式。'J'（JSON, 省略値）/'X'（XML）/'C'（CSV）等。
  - callback: JSONP 用のコールバック関数名（JSONP を使う場合のみ）。

注: パラメータは None または空文字の場合リクエストに含めません。
"""
from dotenv import load_dotenv
import os
import requests
import json
from typing import Optional

API_URL = 'https://api.e-stat.go.jp/rest/3.0/app/json/getStatsList'


def _filter_params(d: dict) -> dict:
    return {k: v for k, v in d.items() if v is not None and v != ''}


def get_stats_list(
    app_id: Optional[str],
    lang: Optional[str] = 'J',
    surveyYears: Optional[str] = None,
    openYears: Optional[str] = None,
    statsField: Optional[str] = None,
    statsCode: Optional[str] = None,
    statsNameList: Optional[str] = None,
    searchWord: Optional[str] = None,
    searchKind: Optional[str] = None,
    collectArea: Optional[str] = None,
    explanationGetFlg: Optional[str] = None,
    startPosition: Optional[int] = None,
    limit: Optional[int] = None,
    updatedDate: Optional[str] = None,
    dataFormat: Optional[str] = None,
    callback: Optional[str] = None,
):
    """Call e-stat getStatsList (統計表情報取得).

    All parameters follow the e-stat v3.0 API specification. Any parameter passed as
    None or empty string will be omitted from the request.

    Returns parsed JSON (dict) on success. Raises SystemExit on HTTP/request errors.
    """
    if not app_id:
        raise ValueError('app_id is required (set E_STAT_APPID or APPID in .env)')

    params = {
        'appId': app_id,
        'lang': lang,
        'surveyYears': surveyYears,
        'openYears': openYears,
        'statsField': statsField,
        'statsCode': statsCode,
        'statsNameList': statsNameList,
        'searchWord': searchWord,
        'searchKind': searchKind,
        'collectArea': collectArea,
        'explanationGetFlg': explanationGetFlg,
        'startPosition': startPosition,
        'limit': limit,
        'updatedDate': updatedDate,
        'dataFormat': dataFormat,
        'callback': callback,
    }

    safe_params = _filter_params(params)

    try:
        resp = requests.get(API_URL, params=safe_params, timeout=30)
        resp.raise_for_status()
    except requests.RequestException as e:
        raise SystemExit(f'API request failed: {e}')

    # API JSON endpoint returns JSON structure under GET_STATS_LIST
    try:
        return resp.json()
    except ValueError:
        # fallback: return raw text if JSON parsing fails
        return {'raw': resp.text}


def pretty_print(result: dict, max_chars: int = 4000) -> None:
    s = json.dumps(result, ensure_ascii=False, indent=2)
    print(s[:max_chars])


if __name__ == '__main__':
    # Example usage when run as a script
    load_dotenv()
    app_id = os.getenv('E_STAT_APPID') or os.getenv('APPID')
    if not app_id:
        raise SystemExit('Please set E_STAT_APPID (or APPID) in .env')

    # Minimal example: search for 統計表 containing '人口'
    res = get_stats_list(
        app_id=app_id,
        # searchWord='課税標準額',
        limit=20,
        dataFormat='J',
        statsCode='002005027',
    )
    pretty_print(res)


{
  "GET_STATS_LIST": {
    "RESULT": {
      "STATUS": 102,
      "ERROR_MSG": "政府統計コード（statsCode）の値が正しくありません。",
      "DATE": "2026-06-24T23:37:19.643+09:00"
    },
    "PARAMETER": {
      "LANG": "J",
      "STATS_CODE": "002005027",
      "DATA_FORMAT": "J",
      "LIMIT": 20
    }
  }
}


In [42]:
# 必要ライブラリの読み込み
import pandas as pd

# Pandas 2.2以降で消された applymap を、現行の map に身代わりさせる
if not hasattr(pd.DataFrame, "applymap"):
    pd.DataFrame.applymap = pd.DataFrame.map

import jpstat
from dotenv import load_dotenv
import os
import requests
import json

# .env を読み込む（事前に .env に E_STAT_APPID=xxxxx を設定しておく）
load_dotenv()
app_id = os.getenv('E_STAT_APPID') or os.getenv('APPID') or ''
if not app_id:
    raise EnvironmentError('`.env` に E_STAT_APPID または APPID を設定してください')

stat = jpstat.estat.get_stat(key=app_id)

In [44]:
display(stat)

,@id,STAT_NAME,GOV_ORG
0,00020111,民間企業の勤務条件制度等調査,人事院
1,00020112,国家公務員死因調査,人事院
2,00020131,国家公務員災害補償統計,人事院
3,00020151,退職公務員生活状況調査,人事院
4,00020211,一般職の国家公務員の任用状況調査,人事院
...,...,...,...
289,00650401,家庭からの二酸化炭素排出量の推計に係る実態調査 試験調査,環境省
290,00650402,大気汚染に係る環境保健サーベイランス調査,環境省
291,00650405,食品廃棄物等の発生抑制及び再生利用の促進の取組に係る実態調査,環境省
292,00650408,家庭部門のCO2排出実態統計調査,環境省


In [ ]:
jpstat.options["estat.api_key"] = app_id
data = jpstat.estat.get_data(statsDataId="0000040001", return_note=False)

In [48]:
data

,@unit,産業大分類040001,経営組織040002,全国計040001,時間軸(年次),Value
0,NaN,全産業,総数,全国,1981年,6488329
1,事業所,全産業,民営,全国,1981年,6290703
2,NaN,全産業,個人,全国,1981年,4182274
3,NaN,全産業,法人,全国,1981年,2074479
4,NaN,全産業,会社,全国,1981年,1843464
...,...,...,...,...,...,...
166,NaN,サービス業,公共企業体,全国,1981年,447
167,NaN,サービス業,地方公共団体,全国,1981年,103593
168,NaN,公務,総数,全国,1981年,45765
169,NaN,公務,国,全国,1981年,7841


In [49]:
data = jpstat.estat.get_list(searchWord="人口")

In [50]:
data

,@id,STAT_NAME,GOV_ORG,STATISTICS_NAME,TITLE,SURVEY_DATE,OPEN_DATE,OVERALL_TOTAL_NUMBER
0,0000150041,人口推計,総務省,人口推計 平成5年10月1日現在推計人口,"人口及び人口増加(29),男女別(3)人口数-総人口,日本人人口,外国人人口,全国",199310,2007-10-03,87
1,0000150062,人口推計,総務省,人口推計 平成6年10月1日現在推計人口,"人口及び人口増加(29),男女別(3)人口数-総人口,日本人人口,外国人人口,全国",199410,2007-10-03,87
2,0000150271,人口推計,総務省,人口推計 各年10月1日現在人口 平成17年国勢調査基準 参考表,[参考表]年齢各歳(94)、男女別(3)、人口数、死亡者数、入国超過-総人口、日本人人口(9...,200610,2007-10-03,2538
3,0000150182,人口推計,総務省,人口推計 各年10月1日現在人口 平成12年国勢調査基準 参考表,[参考表]年齢各歳(94)、男女別(3)、人口数、死亡者数、入国超過数-総人口、日本人人口、...,200110,2007-10-03,2538
4,0000150204,人口推計,総務省,人口推計 各年10月1日現在人口 平成12年国勢調査基準 参考表,[参考表]年齢各歳(94)、男女別(3)、人口数、死亡者数、入国超過数-総人口、日本人人口(...,200210,2007-10-03,2538
...,...,...,...,...,...,...,...,...
25011,0003157566,空き家所有者実態調査,国土交通省,平成26年空家実態調査,7 今後5年程度のうちの利用意向・内容及びその理由等 賃貸・売却する上での課題(11区分) ...,201401-201412,2016-12-28,351
25012,0003157567,空き家所有者実態調査,国土交通省,平成26年空家実態調査,7 今後5年程度のうちの利用意向・内容及びその理由等 賃貸・売却する上での課題(11区分) ...,201401-201412,2016-12-28,273
25013,0003157580,空き家所有者実態調査,国土交通省,平成26年空家実態調査,7 今後5年程度のうちの利用意向・内容及びその理由等 賃貸・売却する上での課題(11区分) ...,201401-201412,2016-12-28,273
25014,0003157581,空き家所有者実態調査,国土交通省,平成26年空家実態調査,7 今後5年程度のうちの利用意向・内容及びその理由等 賃貸・売却する上での課題(11区分) ...,201401-201412,2016-12-28,429
